In [1]:
print("Notebook is working")

Notebook is working


In [3]:
import os

os.chdir("../")
print("Current directory:", os.getcwd())

Current directory: c:\Users\Kunal\Music\ML\domain-rag-assistant


In [4]:
from typing import List
from langchain.schema import Document
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [5]:
def load_pdf_files(data_path: str) -> List[Document]:
    """
    Load all PDF files from a given folder.
    """
    loader = DirectoryLoader(
        data_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

### loading medical 

In [6]:
DOMAIN = "medical"
DATA_PATH = f"data/{DOMAIN}"

extracted_data = load_pdf_files(DATA_PATH)

print("Number of PDF pages loaded:", len(extracted_data))

Number of PDF pages loaded: 637


In [7]:
print(extracted_data[0].page_content[:500])
print(extracted_data[0].metadata)


{'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\medical\\Medical_book.pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}


In [8]:
def filter_to_minimal_docs(docs: List[Document], domain: str) -> List[Document]:
    """
    Keep page content and useful metadata for citations and domain filtering.
    """
    minimal_docs: List[Document] = []

    for doc in docs:
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={
                    "source": doc.metadata.get("source"),
                    "page": doc.metadata.get("page"),
                    "domain": domain
                }
            )
        )

    return minimal_docs

In [9]:
minimal_docs = filter_to_minimal_docs(extracted_data, DOMAIN)

print(minimal_docs[0].metadata)

{'source': 'data\\medical\\Medical_book.pdf', 'page': 0, 'domain': 'medical'}


#### splititng into chunks

In [10]:
def text_split(docs: List[Document]) -> List[Document]:
    """
    Split documents into smaller chunks for RAG retrieval.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    text_chunks = text_splitter.split_documents(docs)
    return text_chunks

In [11]:
texts_chunk = text_split(minimal_docs)

print("Number of chunks:", len(texts_chunk))
print(texts_chunk[0].page_content[:300])
print(texts_chunk[0].metadata)

Number of chunks: 5859
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
{'source': 'data\\medical\\Medical_book.pdf', 'page': 1, 'domain': 'medical'}


#### Adding chunk IDs

In [12]:
def add_chunk_ids(chunks: List[Document]) -> List[Document]:
    """
    Add a unique chunk_id to every chunk.
    """
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    return chunks

In [13]:
texts_chunk = add_chunk_ids(texts_chunk)

print(texts_chunk[0].metadata)

{'source': 'data\\medical\\Medical_book.pdf', 'page': 1, 'domain': 'medical', 'chunk_id': 0}


In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )

    return embeddings

In [15]:
embedding = download_embeddings()

test_vector = embedding.embed_query("Hello world")
print("Vector length:", len(test_vector))

Vector length: 384


In [16]:
test_vector

[-0.034477293491363525,
 0.031023185700178146,
 0.006734919268637896,
 0.026108955964446068,
 -0.03936200588941574,
 -0.16030244529247284,
 0.06692398339509964,
 -0.00644145580008626,
 -0.047450482845306396,
 0.014758873730897903,
 0.07087531685829163,
 0.05552761256694794,
 0.019193336367607117,
 -0.026251327246427536,
 -0.010109526105225086,
 -0.026940451934933662,
 0.02230745740234852,
 -0.02222668007016182,
 -0.14969263970851898,
 -0.017492998391389847,
 0.007676251698285341,
 0.05435226485133171,
 0.003254401497542858,
 0.031725890934467316,
 -0.08462139964103699,
 -0.029405971989035606,
 0.051595598459243774,
 0.04812406003475189,
 -0.003314854810014367,
 -0.05827920511364937,
 0.04196922481060028,
 0.022210687398910522,
 0.1281888335943222,
 -0.02233893983066082,
 -0.011656275019049644,
 0.06292839348316193,
 -0.032876357436180115,
 -0.0912260189652443,
 -0.03117534890770912,
 0.05269956961274147,
 0.04703487083315849,
 -0.08420306444168091,
 -0.030056191608309746,
 -0.020744830

In [17]:
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is missing from .env")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is missing from .env")

#### Connecting to Pinecone

In [18]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

pc

#### Creating Pinecone index

In [19]:
from pinecone import ServerlessSpec

index_name = "domain-rag-assistant"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

print("Index ready:", index_name)

Index ready: domain-rag-assistant


#### Uploading chunks to Pinecone

In [20]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

print("Documents uploaded to Pinecone")

Documents uploaded to Pinecone


#### Load existing Pinecone index

In [21]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

print("Connected to existing Pinecone index")

Connected to existing Pinecone index


#### to add more data to existing picone index

In [22]:
# dswith = Document(
#     page_content="This is a test document about machine learning in healthcare.",
#     metadata={
#         "source": "test.pdf",
        
#     }
#     )

# docsearch.add_documents(documents=  [dswith])

In [44]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {"domain": DOMAIN}
    }
)

print("Retriever ready")

Retriever ready


In [24]:
retrieved_docs = retriever.invoke("What is acne?")

print("Number of docs retrieved:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Doc {i} ---")
    print("Metadata:", doc.metadata)
    print("Content:", doc.page_content[:500])

Number of docs retrieved: 5

--- Retrieved Doc 1 ---
Metadata: {'chunk_id': 299.0, 'domain': 'medical', 'page': 39.0, 'source': 'data\\medical\\Medical_book.pdf'}
Content: GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26

--- Retrieved Doc 2 ---
Metadata: {'chunk_id': 299.0, 'domain': 'medical', 'page': 39.0, 'source': 'data\\medical\\Medical_book.pdf'}
Content: GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26

--- Retrieved Doc 3 ---
Metadata: {'chunk_id': 289.0, 'domain': 'medical', 'page': 38.0, 'source': 'data\\medical\\Medical_book.pdf'}
Content: GALE ENCYCLOPEDIA OF MEDICINE 2 25
Acne
Acne vulgaris affecting a woman’s face. Acne is the general
name given to a skin disorder in which the sebaceous
glands become inflamed. (Photograph by Biophoto Associ-
ates, Photo Researchers, Inc. Reproduced by permission.)
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25

--- Retrieved Doc 4 ---
Metadata: {'chunk_id'

In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

GOOGLE_API_KEY2 = os.getenv("GOOGLE_API_KEY2")
GOOGLE_API_KEY3 = os.getenv("GOOGLE_API_KEY3")

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY2,
    temperature=0.2
)

In [40]:
system_prompt = (
    "You are a helpful medical information assistant. "
    "Use only the retrieved context to answer the user's question. "
    "Do not make up information. "
    "If the answer is not available in the context, say you do not know based on the provided documents. "
    "Do not provide a diagnosis or replace professional medical advice. "
    "Answer all parts of the user's question. "
    "If the question mentions two or more medical terms, explain each term and compare them if possible. "
    "If the question asks about a condition, include: what it is, main cause if available, key features/symptoms if available, and how it differs from related conditions if mentioned. "
    "Keep the answer clear, simple, and concise. "
    "At the end, remind the user to consult a qualified healthcare professional for serious or urgent concerns."
    "\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

print("Prompt ready")

Prompt ready


In [41]:
question_answer_chain = create_stuff_documents_chain(
    chatModel,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

print("RAG chain ready")

RAG chain ready


In [42]:
response = rag_chain.invoke({
    "input": "what is small pox?"
})

print(response["answer"])

I do not know what smallpox is based on the provided documents.

Please consult a qualified healthcare professional for serious or urgent concerns.


In [43]:
response = rag_chain.invoke({
    "input": "Explain the difference between acromegaly and gigantism."
})

print(response["answer"])

seen_sources = set()

print("\nSources:")
for doc in response["context"]:
    source = doc.metadata.get("source", "Unknown source")
    page = doc.metadata.get("page", "Unknown page")
    chunk_id = doc.metadata.get("chunk_id", "Unknown chunk")

    source_key = (source, page, chunk_id)

    if source_key not in seen_sources:
        seen_sources.add(source_key)
        print(f"- {source}, page {int(page)}, chunk {int(chunk_id)}")
        

Based on the provided documents:

**Acromegaly** is a disorder caused by the abnormal release of a specific chemical from the pituitary gland in the brain. This leads to increased growth in bone and soft tissue, along with various other disturbances throughout the body.

The provided context mentions "Acromegaly and gigantism" together but does not offer a definition or explanation for gigantism, nor does it explain the difference between the two conditions.

Please consult a qualified healthcare professional for serious or urgent concerns.

Sources:
- data\medical\Medical_book.pdf, page 45, chunk 356
- data\medical\Medical_book.pdf, page 47, chunk 376
- data\medical\Medical_book.pdf, page 46, chunk 369
